# MCDI505 — Análisis de Series de Tiempo
## Sumativa 1 — Plantilla de apoyo para análisis exploratorio

**Estudiante:** [COMPLETAR]  
**Fecha:** [COMPLETAR]  
**Dataset:** `consumo_electrico_litoral.csv`

> **Declaración de uso de IA:** Se utilizó IA generativa únicamente como apoyo para estructurar el notebook, organizar el código y mejorar la presentación. Las interpretaciones, decisiones metodológicas y conclusiones marcadas como **[RESPUESTA DEL ESTUDIANTE]** deben ser elaboradas y verificadas por el estudiante, conforme a las instrucciones de la evaluación.

## 1. Introducción y objetivo

Este notebook organiza el análisis exigido por la rúbrica: (1) exploración y visualización, (2) descomposición e identificación de componentes, (3) ACF y PACF, y (4) interpretación técnica integrada.

**[RESPUESTA DEL ESTUDIANTE]** Redacte aquí, en 1–2 párrafos, el objetivo del análisis y el contexto del dataset. No copie una respuesta generada automáticamente: explique con sus palabras qué representa la serie, su frecuencia, periodo y variable objetivo.

## 2. Configuración del entorno y carga reproducible

La celda siguiente intenta encontrar el CSV directamente o extraerlo desde `f1_s01_dataset.zip`. Esto permite ejecutar el notebook desde una carpeta limpia, siempre que el dataset o el ZIP estén junto al notebook.

In [ ]:
from pathlib import Path
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["axes.grid"] = True

# Buscar el CSV en ubicaciones habituales.
candidatos = [
    Path("consumo_electrico_litoral.csv"),
    Path("f1_dataset/consumo_electrico_litoral.csv"),
]

csv_path = next((p for p in candidatos if p.exists()), None)

if csv_path is None:
    zip_path = Path("f1_s01_dataset.zip")
    if not zip_path.exists():
        raise FileNotFoundError(
            "No se encontró consumo_electrico_litoral.csv ni f1_s01_dataset.zip. "
            "Ubique uno de ellos en la misma carpeta del notebook."
        )
    destino = Path("f1_dataset")
    destino.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(destino)
    csv_path = destino / "consumo_electrico_litoral.csv"

print(f"Archivo utilizado: {csv_path.resolve()}")

In [ ]:
# Carga y normalización del índice temporal.
df = pd.read_csv(csv_path, parse_dates=["fecha"])
df = df.sort_values("fecha").set_index("fecha")
df.index = pd.DatetimeIndex(df.index)
df = df.asfreq("MS")

print(f"Dimensiones: {df.shape}")
display(df.head())
display(df.tail())

## 3. Contexto y auditoría de calidad de datos

Antes de analizar una serie temporal se verifica la integridad del índice, frecuencia, duplicados, valores faltantes y estadística descriptiva. Esta etapa evita interpretar artefactos producidos por problemas de calidad.

In [ ]:
print("Periodo:", df.index.min().date(), "a", df.index.max().date())
print("Frecuencia inferida:", pd.infer_freq(df.index))
print("Observaciones:", len(df))
print("Fechas duplicadas:", df.index.duplicated().sum())
print("Valores faltantes por variable:")
display(df.isna().sum().to_frame("n_faltantes"))

display(df.describe().T)

faltantes = df.index[df["consumo_gwh"].isna()]
print("Meses con consumo faltante:", [d.strftime("%Y-%m") for d in faltantes])

### 3.1 Decisión de tratamiento de faltantes

La serie contiene observaciones faltantes y debe definirse una estrategia antes de la descomposición y ACF/PACF. La siguiente celda compara alternativas simples sin decidir por el estudiante.

**[RESPUESTA DEL ESTUDIANTE]** Indique qué método utilizará y justifique por qué es razonable para huecos mensuales aislados. Comente también el riesgo de la imputación elegida.

In [ ]:
s = df["consumo_gwh"].copy()
comparacion = pd.DataFrame({
    "original": s,
    "lineal": s.interpolate("linear"),
    "temporal": s.interpolate("time"),
})

# Alternativa estacional: mismo mes del año anterior/siguiente cuando exista.
est = s.copy()
for fecha in faltantes:
    vals = []
    for delta in (-12, 12):
        pos = df.index.get_loc(fecha) + delta
        if 0 <= pos < len(df):
            v = s.iloc[pos]
            if pd.notna(v): vals.append(v)
    if vals:
        est.loc[fecha] = np.mean(vals)
comparacion["estacional_vecinos"] = est

display(comparacion.loc[faltantes])

In [ ]:
# IMPORTANTE: cambie esta línea si su decisión metodológica es otra.
y = s.interpolate("time")
y.name = "consumo_gwh"

assert y.notna().all(), "La serie analítica aún contiene faltantes."
print("Serie analítica preparada:", len(y), "observaciones mensuales")

## 4. Criterio 1 — Exploración y visualización de la serie

La rúbrica exige gráficos completos y una descripción precisa de tendencia, estacionalidad, ciclos, cambios de nivel y/o varianza. Se presentan varias vistas complementarias para que la interpretación no dependa de un único gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y.index, y, lw=1.4, label="Consumo GWh")
ax.plot(y.rolling(12, center=True).mean(), lw=2.2, label="Media móvil 12 meses")
ax.scatter(faltantes, y.loc[faltantes], s=55, zorder=4, label="Valores imputados")
ax.set(title="Consumo eléctrico comercial mensual", xlabel="Fecha", ylabel="Consumo (GWh)")
ax.legend()
plt.show()

In [ ]:
# Perfil estacional mensual y dispersión anual.
perfil_mes = y.groupby(y.index.month).agg(["mean", "std", "median"])
perfil_mes.index.name = "mes"
display(perfil_mes.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(perfil_mes.index, perfil_mes["mean"])
axes[0].set(title="Promedio por mes", xlabel="Mes", ylabel="GWh")
axes[0].set_xticks(range(1,13))

year_df = pd.DataFrame({"año": y.index.year, "consumo": y.values})
year_df.boxplot(column="consumo", by="año", ax=axes[1], rot=45)
axes[1].set(title="Distribución anual", xlabel="Año", ylabel="GWh")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Diagnóstico de relación nivel-varianza para apoyar la elección aditiva/multiplicativa.
roll_mean = y.rolling(12).mean()
roll_std = y.rolling(12).std()
valid = roll_mean.notna() & roll_std.notna()
corr_nivel_disp = roll_mean[valid].corr(roll_std[valid])
print(f"Correlación media móvil vs desviación móvil: {corr_nivel_disp:.3f}")

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(roll_mean, label="Media móvil 12m")
ax.plot(roll_std, label="Desv. estándar móvil 12m")
ax.set(title="Evolución del nivel y la dispersión", xlabel="Fecha")
ax.legend()
plt.show()

### 4.1 Interpretación exploratoria

**[RESPUESTA DEL ESTUDIANTE]** Describa con evidencia visible:

- Tendencia: dirección y posibles cambios.
- Estacionalidad: meses/patrón que se repite y periodicidad aproximada.
- Ciclos o perturbaciones: periodos prolongados que no correspondan solo a la estacionalidad.
- Cambios de nivel o varianza: indique si existen y dónde.
- Calidad de datos: explique cómo los faltantes/imputación afectan la lectura.

Use cifras o periodos concretos de los outputs anteriores; evite frases genéricas.

## 5. Criterio 2 — Descomposición e identificación de componentes

Se utiliza **STL** con periodo 12, adecuado para una serie mensual con estacionalidad anual y robusto para explorar tendencia, componente estacional y residuo. La elección definitiva y su justificación deben relacionarse con la evidencia observada en la sección anterior.

In [ ]:
# STL sobre nivel. Si su decisión justifica transformación logarítmica,
# puede reemplazar y por np.log(y) y explicar cómo interpretar los componentes.
stl = STL(y, period=12, robust=True)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(13, 8)
plt.tight_layout()
plt.show()

In [ ]:
componentes = pd.DataFrame({
    "observada": y,
    "tendencia": res.trend,
    "estacional": res.seasonal,
    "residuo": res.resid,
})

display(componentes.describe().T.round(3))

# Fuerza descriptiva de tendencia y estacionalidad (Hyndman-style, acotada 0..1).
var_r = np.var(res.resid, ddof=1)
var_rt = np.var(res.resid + res.trend, ddof=1)
var_rs = np.var(res.resid + res.seasonal, ddof=1)
f_tend = max(0, 1 - var_r / var_rt) if var_rt > 0 else np.nan
f_est = max(0, 1 - var_r / var_rs) if var_rs > 0 else np.nan
print(f"Fuerza descriptiva de tendencia: {f_tend:.3f}")
print(f"Fuerza descriptiva de estacionalidad: {f_est:.3f}")

### 5.1 Interpretación de la descomposición

**[RESPUESTA DEL ESTUDIANTE]** Explique:

1. Por qué eligió STL, aditivo o multiplicativo a partir de la relación nivel–varianza y del gráfico original.
2. Cómo evoluciona la **tendencia**.
3. Cómo se manifiesta la **estacionalidad** y cuál es su periodicidad.
4. Qué entiende por **ciclo** en este caso y qué evidencia encuentra (si la hay) más allá de la tendencia y estacionalidad.
5. Qué muestra el **error/residuo**: magnitud, episodios anómalos y si parece quedar estructura temporal.

No confunda el residuo con el ciclo: STL entrega tendencia, estacionalidad y residuo; un ciclo de mediano plazo debe inferirse cuidadosamente a partir de la dinámica temporal restante/tendencia suavizada y del contexto.

## 6. Criterio 3 — Análisis de autocorrelación (ACF y PACF)

ACF mide correlación entre la serie y sus rezagos; PACF mide la correlación parcial de un rezago controlando los rezagos intermedios. Para una serie mensual se inspeccionan hasta 48 rezagos (4 años), prestando especial atención a 12, 24, 36 y 48.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
plot_acf(y, lags=48, ax=axes[0], alpha=0.05, zero=False)
plot_pacf(y, lags=48, ax=axes[1], alpha=0.05, zero=False, method="ywm")
axes[0].set_title("ACF — serie en nivel")
axes[1].set_title("PACF — serie en nivel")
plt.tight_layout()
plt.show()

In [ ]:
# Tabla reproducible de ACF/PACF y umbral aproximado 95%: ±1.96/sqrt(N).
NLAGS = 48
acf_vals = acf(y, nlags=NLAGS, fft=True)
pacf_vals = pacf(y, nlags=NLAGS, method="ywm")
umbral = 1.96 / np.sqrt(len(y))

tabla_corr = pd.DataFrame({
    "lag": np.arange(1, NLAGS + 1),
    "ACF": acf_vals[1:],
    "PACF": pacf_vals[1:],
})
tabla_corr["ACF_signif_aprox"] = tabla_corr["ACF"].abs() > umbral
tabla_corr["PACF_signif_aprox"] = tabla_corr["PACF"].abs() > umbral

print(f"Umbral aproximado 95%: ±{umbral:.3f}")
display(tabla_corr.round(3))
print("Rezagos ACF significativos (aprox.):", tabla_corr.loc[tabla_corr.ACF_signif_aprox, "lag"].tolist())
print("Rezagos PACF significativos (aprox.):", tabla_corr.loc[tabla_corr.PACF_signif_aprox, "lag"].tolist())

### 6.1 Interpretación ACF/PACF

**[RESPUESTA DEL ESTUDIANTE]** Responda con base en los gráficos y la tabla:

- ¿La ACF decae lentamente o rápidamente? ¿Qué sugiere sobre persistencia/no estacionariedad?
- ¿Aparecen picos en múltiplos de 12? Relaciónelos con la estacionalidad anual.
- ¿Qué rezagos PACF destacan y qué dependencia directa sugieren?
- ¿ACF y PACF son coherentes con la exploración y la descomposición?
- ¿Qué transformaciones/diferenciaciones evaluaría antes de un modelo ARIMA/SARIMA? Justifique, sin ajustar todavía el modelo final.

**Nota:** el umbral `±1.96/sqrt(N)` es una referencia aproximada. La interpretación principal debe considerar las bandas de confianza mostradas por `statsmodels` y el patrón conjunto, no solo una lista mecánica de rezagos.

## 7. Análisis complementario: ACF/PACF de una serie transformada

Esta sección es diagnóstica. Una diferencia regular puede reducir tendencia y una diferencia estacional de 12 meses puede reducir estacionalidad. Comparar los correlogramas ayuda a entender qué estructura proviene de esos componentes.

In [ ]:
y_diff = y.diff().dropna()
y_sdiff = y.diff(12).dropna()
y_both = y.diff().diff(12).dropna()

fig, axes = plt.subplots(3, 2, figsize=(14, 11))
for row, (serie, nombre) in enumerate([
    (y_diff, "Primera diferencia"),
    (y_sdiff, "Diferencia estacional (12)"),
    (y_both, "Diferencia regular + estacional"),
]):
    max_lags = min(36, len(serie)//2 - 1)
    plot_acf(serie, lags=max_lags, ax=axes[row,0], zero=False)
    plot_pacf(serie, lags=max_lags, ax=axes[row,1], zero=False, method="ywm")
    axes[row,0].set_title(f"ACF — {nombre}")
    axes[row,1].set_title(f"PACF — {nombre}")
plt.tight_layout()
plt.show()

### 7.1 Lectura comparativa

**[RESPUESTA DEL ESTUDIANTE]** Compare la serie original con las versiones diferenciadas. Explique cuál transformación parece remover mejor la persistencia/tendencia y cuál reduce el patrón estacional. No seleccione órdenes ARIMA únicamente por inspección visual: formule una hipótesis para el modelamiento posterior.

## 8. Criterio 4 — Interpretación técnica integrada

**[RESPUESTA DEL ESTUDIANTE]** Redacte una conclusión técnica de 2–4 párrafos que conecte explícitamente:

1. Evidencia exploratoria.
2. Evidencia de la descomposición.
3. Evidencia ACF/PACF.
4. Implicancias para el modelamiento posterior.

Una conclusión de nivel excelente no enumera resultados aislados: explica cómo se corroboran o contradicen entre sí. Indique también las limitaciones del análisis (imputación, eventos atípicos, posible cambio estructural, etc.).

## 9. Checklist de validación contra la rúbrica

Antes de entregar, complete esta tabla manualmente.

| Criterio | Evidencia incluida | Verificación del estudiante |
|---|---|---|
| Exploración y visualización | Serie completa, media móvil, perfil mensual, dispersión anual, nivel-varianza | [ ] |
| Descomposición | STL + justificación del esquema + tendencia/estacionalidad/ciclo/error | [ ] |
| ACF/PACF | Gráficos, tabla de rezagos, significancia e interpretación temporal | [ ] |
| Interpretación integrada | Conclusión que conecta exploración + STL + ACF/PACF + modelamiento | [ ] |
| Redacción | Texto académico revisado, coherente y sin errores relevantes | [ ] |
| Formato | 10–15 páginas equivalentes, nombre solicitado, referencias APA, ejecución completa | [ ] |
| Reproducibilidad | Kernel reiniciado + Run All sin errores | [ ] |
| Uso de IA | Uso declarado y limitado según instrucciones | [ ] |

## 10. Referencias

Complete y ajuste según las fuentes efectivamente consultadas. Ejemplos de referencias técnicas en formato APA:

- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Cleveland, R. B., Cleveland, W. S., McRae, J. E., & Terpenning, I. (1990). STL: A seasonal-trend decomposition procedure based on loess. *Journal of Official Statistics, 6*(1), 3–73.
- Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96.

**[RESPUESTA DEL ESTUDIANTE]** Agregue aquí las lecturas del curso y cualquier otra fuente realmente utilizada, cuidando consistencia APA.

## 11. Control final de ejecución

Ejecute **Kernel → Restart & Run All**. El notebook debe finalizar sin errores. Luego revise que las interpretaciones **[RESPUESTA DEL ESTUDIANTE]** estén completas, que las figuras tengan títulos/ejes legibles y que el nombre final cumpla el patrón solicitado por la evaluación: `mcdi505_s1inicialapellido.ipynb`.